In [ ]:
import hashlib
import json
import os
import random
import time
from datetime import datetime, timezone
from pathlib import Path

from tqdm.auto import tqdm

try:
    from openai import OpenAI
except ImportError as exc:
    raise ImportError("pip install openai") from exc


In [ ]:
# ============================== КОНФИГ ==============================

CONFIG = {
    # "A" — код GitHub как протокол (шкала [0,1]);  "B" — 3 критерия 1-5 (рукопись)
    "JUDGE_VARIANT": "B",

    # ТОЧНЫЙ API model/deployment ID, не просто "GPT-4". Проверить фактически
    # доступную модель в вашем аккаунте перед запуском.
    "MODEL_ID": "gpt-4o-2024-08-06",  # <-- ЗАМЕНИТЬ на реально используемую модель

    "TEMPERATURE": 0.2,
    "N_RUNS": 3,               # ровно 3 независимых вызова на выход
    "ORDER_SEED": 42,          # seed для рандомизации порядка Static/Dynamic
    "MAX_RETRIES": 3,          # retry только при технической ошибке / невалидном JSON
    "RETRY_BACKOFF_SEC": 2.0,

    "STATIC_PREDICTIONS": "results_1000/static_predictions_1000.jsonl",
    "DYNAMIC_PREDICTIONS": "results_1000/dynamic_predictions_1000.jsonl",
    "BENCHMARK": "CMES_evaluation/benchmark_gold_1000.jsonl",

    "OUTPUT_DIR": "results_1000",
    "JUDGE_SCRIPT_VERSION": "gpt_judge_v1_2026-07-28",  # обновляйте при правках скрипта
}

assert CONFIG["JUDGE_VARIANT"] in ("A", "B")
assert os.environ.get("OPENAI_API_KEY"), "Установите OPENAI_API_KEY через переменную среды."

client = OpenAI()  # берёт ключ из OPENAI_API_KEY


In [ ]:
# ============================ УТИЛИТЫ ================================

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def load_jsonl(path: str) -> list[dict]:
    rows = []
    with open(path, encoding="utf-8-sig") as fh:
        for line in fh:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def save_jsonl(path: str, rows: list[dict]) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="\n") as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def generated_table_of(row: dict) -> str:
    for key in ("generated_table", "generated", "table"):
        value = row.get(key)
        if isinstance(value, str) and value.strip():
            return value
    raise ValueError(f"id {row.get('id')!r}: no generated table field found")


In [ ]:
# ===================== ПРОМПТЫ: VARIANT A (заглушка) =====================
# ВНИМАНИЕ: скопировать точный текст из src/evaluation/llm_judge.py
# (commit ee537aaf1d3395b69f43f85998cbf0c8738aa636) перед запуском Variant A.
# Не запускайте Variant A, пока эта заглушка не заменена реальным промптом —
# иначе заявление "использовали протокол репозитория" будет недостоверным.

_VARIANT_A_SYSTEM_PROMPT_TODO = (
    "TODO: вставьте сюда точный system prompt из llm_judge.py. "
    "НЕ запускать с этой заглушкой."
)

def build_prompt_variant_a(source_text: str, generated_table: str) -> tuple[str, str]:
    if _VARIANT_A_SYSTEM_PROMPT_TODO.startswith("TODO"):
        raise RuntimeError(
            "Variant A: промпт-заглушка не заменена реальным кодом из llm_judge.py."
        )
    system_prompt = _VARIANT_A_SYSTEM_PROMPT_TODO
    user_prompt = f"SOURCE:\n{source_text}\n\nTABLE:\n{generated_table}"
    return system_prompt, user_prompt

def parse_response_variant_a(raw_text: str) -> dict:
    payload = json.loads(raw_text)
    value = float(payload["journalistic_value"])
    if not (0.0 <= value <= 1.0):
        raise ValueError(f"journalistic_value вне диапазона [0,1]: {value}")
    return {"journalistic_value": value, "raw_scores": payload}

def aggregate_variant_a(parsed: dict) -> float:
    return parsed["journalistic_value"]


In [ ]:
# ===================== ПРОМПТЫ: VARIANT B (готово) =====================

_VARIANT_B_SYSTEM_PROMPT = (
    "You are an expert Kazakh-language journalism editor evaluating machine-"
    "generated summary tables produced from news source text. You will be "
    "shown the ORIGINAL SOURCE TEXT and one GENERATED TABLE. Score the "
    "GENERATED TABLE on exactly three criteria, each on an integer scale "
    "from 1 (very poor) to 5 (excellent):\n"
    "1. faithfulness — factual consistency of table entries with the source text, "
    "no invented facts;\n"
    "2. coverage_completeness — how completely the table captures the key facts "
    "of the source text (who/what/when/where/numbers);\n"
    "3. journalistic_usefulness — how useful/publishable this table would be "
    "for a professional journalist writing a news piece.\n\n"
    "Respond with STRICT JSON ONLY, no markdown fences, no commentary, in "
    "exactly this schema:\n"
    '{"criterion_1_faithfulness": <int 1-5>, '
    '"criterion_2_coverage": <int 1-5>, '
    '"criterion_3_usefulness": <int 1-5>}'
)

def build_prompt_variant_b(source_text: str, generated_table: str) -> tuple[str, str]:
    # Слово "static"/"dynamic"/regime намеренно НЕ передаётся модели.
    user_prompt = (
        f"SOURCE TEXT:\n{source_text}\n\n"
        f"GENERATED TABLE:\n{generated_table}\n\n"
        "Return the JSON scores now."
    )
    return _VARIANT_B_SYSTEM_PROMPT, user_prompt

def parse_response_variant_b(raw_text: str) -> dict:
    payload = json.loads(raw_text)
    scores = {}
    for key in ("criterion_1_faithfulness", "criterion_2_coverage", "criterion_3_usefulness"):
        r = int(payload[key])
        if r not in (1, 2, 3, 4, 5):
            raise ValueError(f"{key} вне диапазона 1-5: {r}")
        scores[key] = r
    return {"raw_scores": scores}

def aggregate_variant_b(parsed: dict) -> float:
    # (r-1)/4 нормировка каждого критерия, затем среднее трёх -> одно
    # journalistic_value на вызов. См. допущение в markdown-ячейке выше.
    normalized = [(v - 1) / 4 for v in parsed["raw_scores"].values()]
    return sum(normalized) / len(normalized)


In [ ]:
# ============================ API-ВЫЗОВ ================================

def call_judge_once(system_prompt: str, user_prompt: str) -> dict:
    """Один вызов Chat Completions API. Возвращает сырой ответ + метаданные.
    Поднимает исключение при технической ошибке / невалидном JSON — верхний
    уровень отвечает за retry-логику и её журналирование."""
    response = client.chat.completions.create(
        model=CONFIG["MODEL_ID"],
        temperature=CONFIG["TEMPERATURE"],
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    raw_text = response.choices[0].message.content
    json.loads(raw_text)  # провалидировать, что это валидный JSON; иначе исключение -> retry
    return {
        "raw_text": raw_text,
        "response_id": getattr(response, "id", None),
        "usage": dict(response.usage) if getattr(response, "usage", None) else None,
        "model_returned": response.model,
    }

def call_judge_with_retries(system_prompt: str, user_prompt: str) -> dict:
    last_error = None
    for attempt in range(1, CONFIG["MAX_RETRIES"] + 1):
        try:
            result = call_judge_once(system_prompt, user_prompt)
            result["retries"] = attempt - 1
            result["error"] = None
            return result
        except Exception as exc:  # технический сбой или невалидный JSON -> разрешён retry
            last_error = exc
            time.sleep(CONFIG["RETRY_BACKOFF_SEC"] * attempt)
    return {
        "raw_text": None,
        "response_id": None,
        "usage": None,
        "model_returned": None,
        "retries": CONFIG["MAX_RETRIES"],
        "error": f"{type(last_error).__name__}: {last_error}",
    }


In [ ]:
# ===================== ПОСТРОЕНИЕ ОЧЕРЕДИ ЗАДАЧ =====================
# Рандомизация и журналирование порядка Static/Dynamic (Раздел 7, пункт
# "рандомизирует и журналирует порядок"). Слово static/dynamic никогда не
# передаётся модели в промпте — используется только для внутреннего лога.

def build_task_queue(benchmark: list[dict], static: list[dict], dynamic: list[dict]) -> list[dict]:
    static_by_id = {str(r["id"]): r for r in static}
    dynamic_by_id = {str(r["id"]): r for r in dynamic}
    bench_by_id = {str(r["id"]): r for r in benchmark}

    ids = sorted(bench_by_id)
    assert set(ids) == set(static_by_id) == set(dynamic_by_id), "ID Gold/Static/Dynamic не совпадают"

    tasks = []
    for record_id in ids:
        for regime, source in (("static", static_by_id), ("dynamic", dynamic_by_id)):
            for run_index in (1, 2, 3):
                tasks.append({
                    "id": record_id,
                    "hidden_regime": regime,
                    "run_index": run_index,
                    "source_text": bench_by_id[record_id]["text"],
                    "generated_table": generated_table_of(source[record_id]),
                })

    rng = random.Random(CONFIG["ORDER_SEED"])
    order = list(range(len(tasks)))
    rng.shuffle(order)
    shuffled = [tasks[i] for i in order]
    for position, task in enumerate(shuffled, 1):
        task["queue_position"] = position
        task["shown_label"] = rng.choice(["A", "B"])  # blind label logged, never sent as static/dynamic
    return shuffled


In [ ]:
# ============================ SMOKE TEST ============================
# Прогнать на 3 записях перед полным запуском (2000 * 3 = 6000 вызовов).

benchmark = load_jsonl(CONFIG["BENCHMARK"])
static_preds = load_jsonl(CONFIG["STATIC_PREDICTIONS"])
dynamic_preds = load_jsonl(CONFIG["DYNAMIC_PREDICTIONS"])

smoke_ids = {str(r["id"]) for r in benchmark[:3]}
smoke_benchmark = [r for r in benchmark if str(r["id"]) in smoke_ids]
smoke_static = [r for r in static_preds if str(r["id"]) in smoke_ids]
smoke_dynamic = [r for r in dynamic_preds if str(r["id"]) in smoke_ids]

smoke_tasks = build_task_queue(smoke_benchmark, smoke_static, smoke_dynamic)
print(f"Smoke queue size: {len(smoke_tasks)} (ожидается 3 id * 2 regime * 3 runs = 18)")

build_prompt, parse_response, aggregate = {
    "A": (build_prompt_variant_a, parse_response_variant_a, aggregate_variant_a),
    "B": (build_prompt_variant_b, parse_response_variant_b, aggregate_variant_b),
}[CONFIG["JUDGE_VARIANT"]]

smoke_raw = []
for task in tqdm(smoke_tasks, desc="smoke"):
    system_prompt, user_prompt = build_prompt(task["source_text"], task["generated_table"])
    api_result = call_judge_with_retries(system_prompt, user_prompt)
    parsed = None
    jv = None
    if api_result["error"] is None:
        try:
            parsed = parse_response(api_result["raw_text"])
            jv = aggregate(parsed)
        except Exception as exc:
            api_result["error"] = f"parse_error: {exc}"
    smoke_raw.append({**task, **api_result, "parsed": parsed, "journalistic_value": jv,
                       "system_prompt_sha256": sha256_text(system_prompt),
                       "user_prompt_sha256": sha256_text(user_prompt),
                       "timestamp_utc": utc_now(),
                       "judge_script_version": CONFIG["JUDGE_SCRIPT_VERSION"],
                       "model_id": CONFIG["MODEL_ID"], "temperature": CONFIG["TEMPERATURE"]})

n_errors = sum(1 for r in smoke_raw if r["error"] is not None)
print(f"Smoke errors: {n_errors} / {len(smoke_raw)}")
smoke_raw[:2]


**Проверьте вручную перед полным запуском:**
- `n_errors == 0` (или все ошибки объяснимы и не системны);
- `journalistic_value` в разумном диапазоне, не все значения одинаковы;
- `system_prompt_sha256` одинаков для всех задач одного варианта (промпт не "плывёт");
- для Variant A: заглушка была заменена реальным промптом (иначе ячейка выше уже упала бы с `RuntimeError`).


In [ ]:
# ======================= ПОЛНЫЙ ЗАПУСК: 6000 ВЫЗОВОВ =======================
# Это может занять долго и стоить денег (API). Запускать только после smoke test.

full_tasks = build_task_queue(benchmark, static_preds, dynamic_preds)
assert len(full_tasks) == 1000 * 2 * 3, f"Ожидалось 6000 задач, получено {len(full_tasks)}"

raw_log_path = str(Path(CONFIG["OUTPUT_DIR"]) / "gpt_judge_raw.jsonl")
Path(raw_log_path).parent.mkdir(parents=True, exist_ok=True)

with open(raw_log_path, "w", encoding="utf-8", newline="\n") as fh:
    for task in tqdm(full_tasks, desc="gpt-judge full run"):
        system_prompt, user_prompt = build_prompt(task["source_text"], task["generated_table"])
        api_result = call_judge_with_retries(system_prompt, user_prompt)
        parsed = None
        jv = None
        if api_result["error"] is None:
            try:
                parsed = parse_response(api_result["raw_text"])
                jv = aggregate(parsed)
            except Exception as exc:
                api_result["error"] = f"parse_error: {exc}"
        row = {
            "id": task["id"],
            "hidden_regime": task["hidden_regime"],
            "shown_label": task["shown_label"],
            "run_index": task["run_index"],
            "queue_position": task["queue_position"],
            "model_id": CONFIG["MODEL_ID"],
            "model_returned": api_result["model_returned"],
            "temperature": CONFIG["TEMPERATURE"],
            "system_prompt_sha256": sha256_text(system_prompt),
            "user_prompt_sha256": sha256_text(user_prompt),
            "raw_text": api_result["raw_text"],
            "parsed": parsed,
            "journalistic_value": jv,
            "response_id": api_result["response_id"],
            "usage": api_result["usage"],
            "retries": api_result["retries"],
            "error": api_result["error"],
            "timestamp_utc": utc_now(),
            "judge_script_version": CONFIG["JUDGE_SCRIPT_VERSION"],
            "judge_variant": CONFIG["JUDGE_VARIANT"],
        }
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Готово:", raw_log_path)


In [ ]:
# =============== СБОРКА judge_runs И ФАЙЛОВ *_with_judge.jsonl ===============

raw_rows = load_jsonl(raw_log_path)
errors = [r for r in raw_rows if r["error"] is not None]
print(f"Строк в raw log: {len(raw_rows)}; ошибок (после исчерпания retries): {len(errors)}")
if errors:
    print("ВНИМАНИЕ: есть неустранённые ошибки — исправьте их (например, перезапустив только")
    print("эти id/regime/run) до сборки итоговых файлов, иначе judge_runs будет неполным.")

by_id_regime: dict[tuple[str, str], list[dict]] = {}
for row in raw_rows:
    if row["error"] is not None:
        continue
    key = (row["id"], row["hidden_regime"])
    by_id_regime.setdefault(key, []).append(row)

def build_with_judge(predictions: list[dict], regime: str) -> list[dict]:
    out = []
    for pred in predictions:
        record_id = str(pred["id"])
        runs = sorted(by_id_regime.get((record_id, regime), []), key=lambda r: r["run_index"])
        if len(runs) != 3:
            raise ValueError(f"{regime}/{record_id}: ожидалось 3 judge_runs, найдено {len(runs)}")
        pred_with_judge = dict(pred)
        pred_with_judge["judge_runs"] = [{"journalistic_value": r["journalistic_value"]} for r in runs]
        out.append(pred_with_judge)
    return out

static_with_judge = build_with_judge(static_preds, "static")
dynamic_with_judge = build_with_judge(dynamic_preds, "dynamic")

save_jsonl(str(Path(CONFIG["OUTPUT_DIR"]) / "static_predictions_1000_with_judge.jsonl"), static_with_judge)
save_jsonl(str(Path(CONFIG["OUTPUT_DIR"]) / "dynamic_predictions_1000_with_judge.jsonl"), dynamic_with_judge)

manifest = {
    "created_utc": utc_now(),
    "judge_variant": CONFIG["JUDGE_VARIANT"],
    "model_id": CONFIG["MODEL_ID"],
    "temperature": CONFIG["TEMPERATURE"],
    "runs_per_output": CONFIG["N_RUNS"],
    "order_seed": CONFIG["ORDER_SEED"],
    "judge_script_version": CONFIG["JUDGE_SCRIPT_VERSION"],
    "total_raw_calls": len(raw_rows),
    "total_errors_after_retries": len(errors),
    "expected_calls": 1000 * 2 * 3,
    "raw_log_sha256": sha256_text(Path(raw_log_path).read_text(encoding='utf-8')),
}
manifest_path = str(Path(CONFIG["OUTPUT_DIR"]) / "gpt_judge_manifest.json")
Path(manifest_path).write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(manifest_path)
manifest
